In [0]:
%sql
SELECT * FROM prod.customer_reports.hashes_production_comscore
WHERE hash IN ('5742334b6733326f7348704b716b30625a744358377343707630786d7961384f6b436d38664950706154383d'
, '7035675658596c2f43704b4a7969586a31324b5a576343707630786d7961384f6b436d38664950706154383d'
, '4f2f41384a643269506e58323567496e37334c4e466343707630786d7961384f6b436d38664950706154383d')

In [0]:
%sql
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, vc.fk_show_id
, vc.tuner_program_id
, vc.fk_station_id
, vc.tuner_channel_id
, vc.airdate
, vc.media_time_start
, vc.media_time_end
, vc.is_live
FROM detection.viewing_content_firehose vc
WHERE fk_tvid IN (164224255, 164267145, 175833069)
  AND session_start >= '2025-03-11 08:00:00'
  AND session_start < '2025-03-12 00:00:00'
ORDER BY 1, 2, 3

In [0]:
%sql
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, sh.title
, COALESCE(st.inscape_station_name, st.station_name) as station_name
, vc.airdate
-- , vc.media_time_start
-- , vc.media_time_end
, vc.is_live
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station st
  ON st.station_id = COALESCE(vc.fk_station_id, vc.tuner_channel_id)
LEFT JOIN detection.epg_show sh
  ON sh.show_id = COALESCE(vc.fk_show_id, vc.tuner_program_id)
WHERE vc.fk_tvid IN (164224255, 164267145, 175833069)
AND session_start >= '2025-03-11 00:00:00'
AND session_start < '2025-03-12 00:00:00'
ORDER BY 1, 2, 3

In [0]:
%sql
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
FROM detection.viewing_commercials_firehose vc
WHERE vc.fk_tvid IN (164224255, 164267145, 175833069)
AND session_start >= '2025-03-11 00:00:00'
AND session_start < '2025-03-12 00:00:00'
GROUP BY ALL
ORDER BY 1, 2, 3

In [0]:
%sql
SELECT * FROM detection.epg_station
WHERE station_call_sign LIKE '%KATU%'
AND (ingested = 'TRUE' OR attributed = 'TRUE')
AND vendor_name = 'TIVO'

In [0]:
%sql
SELECT sch.airdate, sh.title
FROM detection.epg_schedule sch
JOIN detection.epg_show sh
  ON sh.show_id = sch.fk_show_id
WHERE sch.airdate >= '2025-03-11 00:00:00'
  AND sch.airdate < '2025-03-12 00:00:00'
  AND sch.fk_station_id = 93563
  AND sch.vendor_name = 'TIVO'
GROUP BY 1, 2

In [0]:
%sql
-- SELECT fk_tvid
-- , title
-- , station_name
-- , call_sign
-- , airdate
-- , runtime
-- , is_live
-- , calc
-- , MIN(session_start) AS session_start
-- , MAX(session_end) AS session_end
-- , SUM(session_duration) AS session_duration
-- , MIN(media_time_start) AS mts
-- , MAX(media_time_end) AS mte
-- FROM (
-- SELECT *
-- , SUM(next_session_match) OVER (PARTITION BY fk_tvid ORDER BY session_start ROWS UNBOUNDED PRECEDING) AS calc
-- FROM (
-- SELECT *
-- , CASE WHEN viewing_type = next_viewing_type 
--         AND next_start = session_end
--         AND title <=> next_title
--         AND station_name <=> next_station_name
--         AND call_sign <=> next_call_sign
--         AND airdate <=> next_airdate
--         AND is_live <=> next_is_live
--         THEN 0 ELSE 1 END AS next_session_match
-- FROM (
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, sh.title
, st.inscape_station_name as station_name
, st.station_call_sign AS call_sign
, vc.airdate
, vc.media_time_start
, vc.media_time_end
, vc.runtime
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
-- , LEAD(vc.session_start) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_start
-- , LEAD(sh.title) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_title
-- , LEAD(st.inscape_station_name) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_station_name
-- , LEAD(st.station_call_sign) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_call_sign
-- , LEAD(vc.airdate) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_airdate
-- , LEAD(vc.is_live) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_is_live
-- , LEAD(CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END) OVER (PARTITION BY vc.fk_tvid ORDER BY vc.session_start) AS next_viewing_type
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station st
  ON st.station_id = COALESCE(vc.fk_station_id, vc.tuner_channel_id)
LEFT JOIN detection.epg_show sh
  ON sh.show_id = COALESCE(vc.fk_show_id, vc.tuner_program_id)
WHERE vc.fk_tvid = 160189140
  AND (
       (vc.session_start >= '2025-03-18 17:00:00' AND vc.session_start < '2025-03-18 22:00:00')
    OR (vc.session_start >= '2025-03-20 18:00:00' AND vc.session_start < '2025-03-20 21:00:00')
    OR (vc.session_start >= '2025-03-25 16:00:00' AND vc.session_start < '2025-03-25 19:00:00')
    )
-- )))
-- GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
-- ORDER BY 8

In [0]:
%sql
(SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, sh.title
, st.inscape_station_name as station_name
, st.station_call_sign AS call_sign
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
, 'Our Log' AS source
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station st
  ON st.station_id = COALESCE(vc.fk_station_id, vc.tuner_channel_id)
LEFT JOIN detection.epg_show sh
  ON sh.show_id = COALESCE(vc.fk_show_id, vc.tuner_program_id)
WHERE vc.fk_tvid = 160189140
  AND (
       (vc.session_start >= '2025-03-18 17:00:00' AND vc.session_start < '2025-03-18 22:00:00')
    OR (vc.session_start >= '2025-03-20 18:00:00' AND vc.session_start < '2025-03-20 21:00:00')
    OR (vc.session_start >= '2025-03-25 16:00:00' AND vc.session_start < '2025-03-25 19:00:00')
    )
)
UNION
(SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, 'Commercial' AS title
, 'Commercial' AS station_name
, 'Commercial' AS call_sign
, NULL AS is_live
, 'Commercial' AS viewing_type
, 'Commercial' AS source
FROM detection.viewing_commercials_firehose vc
WHERE vc.fk_tvid = 160189140
  AND (
       (vc.session_start >= '2025-03-18 17:00:00' AND vc.session_start < '2025-03-18 22:00:00')
    OR (vc.session_start >= '2025-03-20 18:00:00' AND vc.session_start < '2025-03-20 21:00:00')
    OR (vc.session_start >= '2025-03-25 16:00:00' AND vc.session_start < '2025-03-25 19:00:00')
    )
)

In [0]:
%sql
SELECT st.station_call_sign, st.ingested, st.attributed, COUNT(*)
FROM detection.viewing_content_firehose vc
JOIN detection.epg_station st
  ON st.station_id = vc.tuner_channel_id
WHERE vc.session_start >= CURRENT_DATE - 1
  AND st.station_call_sign LIKE 'KGO%'
GROUP BY 1, 2, 3

In [0]:
%sql
SELECT st.station_call_sign, st.ingested, st.attributed, COUNT(*)
FROM detection.viewing_content_firehose vc
JOIN detection.epg_station st
  ON st.station_id = vc.fk_station_id
WHERE vc.session_start >= CURRENT_DATE - 1
  AND st.station_call_sign LIKE 'KGO%'
GROUP BY 1, 2, 3

In [0]:
%sql
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, linear_sh.title AS linear_title
, linear_st.inscape_station_name as linear_station_name
, linear_st.station_call_sign AS    linear_call_sign
, tuner_sh.title AS tuner_title
, tuner_st.inscape_station_name as tuner_station_name
, tuner_st.station_call_sign AS    tuner_call_sign
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
, 'Our Log' AS source
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station linear_st
  ON linear_st.station_id = vc.fk_station_id
LEFT JOIN detection.epg_show linear_sh
  ON linear_sh.show_id = vc.fk_show_id
LEFT JOIN detection.epg_station tuner_st
  ON tuner_st.station_id = vc.tuner_channel_id
LEFT JOIN detection.epg_show tuner_sh
  ON tuner_sh.show_id = vc.tuner_program_id
WHERE vc.fk_tvid = 160189140
  AND (
       (vc.session_start >= '2025-03-18 17:00:00' AND vc.session_start < '2025-03-18 22:00:00')
    OR (vc.session_start >= '2025-03-20 18:00:00' AND vc.session_start < '2025-03-20 21:00:00')
    OR (vc.session_start >= '2025-03-25 16:00:00' AND vc.session_start < '2025-03-25 19:00:00')
    )


In [0]:
%sql
WITH activity_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_activity_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
, viewing_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
, station_distribution_blacklist AS (
    SELECT vendor_station_id as station_id, vendor_name
    FROM prod.detection.station_distribution_obfuscation_overwrite
    WHERE CASE WHEN TRUE = FALSE THEN TRUE
        WHEN False = TRUE THEN FALSE END
)
, inscape_station_map_dedupe AS (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    FROM (
        SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        FROM detection.inscape_station_map) ism
    WHERE ism.rn = 1
)
SELECT /*+ BROADCAST(cid) */  DISTINCT
    COALESCE(tv.long_tvid, tv.vizio_tvid) AS tvid,
    -- c.fk_tvid AS tvid,
    c.session_start,
    c.session_end,
    -- NULLIF(location.zipcode, '') AS zipcode,
    -- REPLACE(dma.dma_name, ',', '') AS dma,
    -- NULLIF(CASE
    -- WHEN c.vizio_epg_station IS NOT NULL THEN
    --     CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in (98989898989898, 9898989898, -1) OR 'TMS' != 'TMS' THEN NULL
    --          ELSE vizio_program.program_tms_id
    --     END
    -- ELSE CASE
    --         WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
    --         WHEN (c.file_ingested = true) THEN COALESCE(md.external_id,SPLIT(cid.content_cid, '_')[0])
    --         ELSE show.database_key
    --     END
    -- END,'') AS epid,
    CASE
    WHEN c.vizio_epg_station IS NOT NULL THEN
        CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in (98989898989898, 9898989898, -1) THEN NULL
            WHEN vizio_program.series_aggregate_title IS NOT NULL AND vizio_program.series_aggregate_title != ''
                THEN vizio_program.series_aggregate_title
                ELSE vizio_program.title END
    WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
    ELSE CASE WHEN c.file_ingested THEN NULL
        WHEN 'comscore' != 'nielsen' THEN REPLACE(COALESCE(show.title, backup_show.title), ',', '')
        ELSE REPLACE(show.title, ',', '') END
    END AS show_title,
    CASE
        WHEN c.vizio_epg_station IS NULL AND acrb.app_name IS NOT NULL THEN NULL
        WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in (98989898989898, 9898989898, -1) THEN NULL
        WHEN c.vizio_epg_station IS NOT NULL THEN COALESCE(c.tms_airdate, c.airdate)
        WHEN (cl.client_id is not null) THEN NULL
        WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
        WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
        WHEN station_blacklist.station_id IS NOT NULL THEN NULL
        WHEN 'comscore' != 'nielsen' THEN COALESCE(c.tms_airdate, c.airdate)
        ELSE c.tms_airdate
    END AS airdate,
    CASE
        WHEN c.vizio_epg_station IS NOT NULL THEN COALESCE(map.inscape_call_sign, backup_map.inscape_call_sign)
        WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
        WHEN station_obfs.station_id IS NOT NULL THEN NULL
        WHEN c.file_ingested = true THEN SPLIT(cid.content_cid, '_')[1]
        WHEN 'comscore' != 'nielsen' THEN COALESCE(map.inscape_call_sign, backup_map.inscape_call_sign)
        WHEN 'comscore' = 'nielsen' AND c.tms_airdate IS NOT NULL THEN map.inscape_call_sign
        ELSE NULL
     END AS call_sign,
    -- CASE
    --     WHEN c.vizio_epg_station IS NULL AND acrb.app_name IS NOT NULL THEN NULL
    --     WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in (98989898989898, 9898989898, -1) THEN NULL
    --     WHEN c.vizio_epg_station IS NOT NULL THEN c.media_time_start
    --     WHEN (cl.client_id is not null) THEN NUll
    --     WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
    --     WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
    --     WHEN 'comscore' != 'nielsen' AND COALESCE(c.tms_airdate, c.airdate) IS NULL THEN NULL
    --     WHEN 'comscore' = 'nielsen' AND c.tms_airdate IS NULL THEN NULL
    --     ELSE LEAST(c.media_time_start, c.runtime)
    -- END AS mts,
    CASE WHEN c.vizio_epg_station IS NOT NULL
    THEN CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in (98989898989898, 9898989898, -1) THEN 'OBFUSCATED'
        ELSE vizio_station.name END
    WHEN acrb.app_name IS NOT NULL THEN NULL
    WHEN (cl.client_id IS NOT NULL) THEN NULL
    ELSE
        CASE
            WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
            WHEN station_obfs.station_id IS NOT NULL THEN NULL
            WHEN 'comscore' = 'nielsen' AND c.tms_airdate IS NULL THEN NULL
            WHEN COALESCE(c.tms_station_id, c.tms_tuner_channel_id) IS NOT NULL THEN
                CASE WHEN (station.inscape_station_name IS NOT NULL) THEN station.inscape_station_name
                     WHEN (LOWER(station.station_affil) LIKE '%affiliate%'
                           OR LOWER(station.station_affil) LIKE '%independent%'
                           OR LOWER(station.station_affil) LIKE '%low power%')
                        THEN station.station_affil ELSE NULL END
            WHEN COALESCE(c.tms_station_id, c.tms_tuner_channel_id) IS NULL AND COALESCE(c.fk_station_id, c.tuner_channel_id) IS NOT NULL THEN
                CASE WHEN (backup_station.inscape_station_name IS NOT NULL) THEN backup_station.inscape_station_name
                     WHEN (LOWER(backup_station.station_affil) LIKE '%affiliate%'
                          OR LOWER(backup_station.station_affil) LIKE '%independent%'
                          OR LOWER(backup_station.station_affil) LIKE '%low power%')
                        THEN backup_station.station_affil ELSE NULL END END
    END AS station_name,
    CASE
        WHEN c.vizio_epg_station IS NOT NULL THEN 't'
        WHEN nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local.station_id, rep_nyc_nat.station_id) IS NULL OR ingest_time IS NOT NULL) THEN NULL
        WHEN COALESCE(NULLIF(c.tms_tuner_channel_id,98989898), c.tuner_channel_id) IS NOT NULL THEN 't'
        WHEN 'comscore' != 'nielsen' AND COALESCE(c.tms_airdate,c.airdate) IS NULL THEN NULL
        WHEN 'comscore' = 'nielsen' AND c.tms_airdate IS NULL THEN NULL
        WHEN inps.input_source = 'DTV' OR tvis.input_device = 'OTA' THEN 't'
    ELSE CASE WHEN c.is_live = TRUE THEN 't' WHEN c.is_live = FALSE THEN 'f' ELSE NULL END
    END AS live,
    -- ip.ip_address AS ip,
    CASE
        WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT')
        THEN 'HD TV'
        WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND cid.content_cid != 'unknown'
        THEN 'HD TV'
        WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name ='WatchFree+' THEN 'APPS'
        ELSE tvis.category
    END AS input_category,
    CASE
        WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
        THEN 'OTA'
        WHEN inps.input_source = 'DTV' THEN 'OTA'
        ELSE tvis.input_device
    END AS input_device,
    CASE
        WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) is not null
            AND (input_device IS NULL OR input_device = 'OTA')
            THEN 'WatchFree+'
        WHEN UPPER(tvis.category) = 'APPS' THEN
        CASE WHEN c.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
            WHEN c.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
            WHEN appb.app_name IS NOT NULL THEN 'OBFUSCATED'
            WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND cid.content_cid <> 'unknown' THEN NULL
            WHEN lower(tis.app_name) = 'unknown' THEN NULL
            ELSE tis.app_name END
        WHEN c.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) is NULL
            THEN 'vMVPD'
        WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV') AND (input_device IS NULL OR input_device = 'OTA')
             AND tis.app_name ='WatchFree+' AND c.is_live = TRUE THEN 'WatchFree+'
    END AS app_name
FROM
    prod.detection.viewing_content_firehose AS c
    JOIN prod.detection.zoo AS z ON c.fk_zoo_id = z.zoo_id
        AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
    JOIN prod.detection.tv AS tv ON c.fk_tvid = tv.tvid
        AND tv.tvid = 160189140
        AND tv.oem = 'VIZIO'  
    JOIN prod.detection.tv_settings AS tv_settings
        ON c.session_start >= tv_settings.create_timestamp
        AND c.session_start < tv_settings.next_create_timestamp
        AND tv_settings.create_timestamp <= '2025-03-25 19:00:00'::timestamp
        AND tv_settings.next_create_timestamp >= '2025-03-18 17:00:00'::timestamp
        AND c.fk_tvid = tv_settings.fk_tvid
        AND tv_settings.fk_tvid = 160189140
    JOIN prod.detection.settings AS settings
        ON tv_settings.fk_settings_id = settings.settings_id
        AND UPPER(settings.country_name) = 'USA'
    JOIN prod.detection.tv_populations AS u
        ON c.fk_tvid = u.fk_tvid
        AND u.fk_tvid = 160189140
    JOIN prod.detection.populations AS pop
        ON u.fk_population_id = pop.population_id
        AND pop.population_name = 'opted_in'
    JOIN prod.detection.location AS location
        ON c.fk_location_id = location.location_id
        AND UPPER(location.country_code) = 'US'
    LEFT OUTER JOIN prod.detection.dma AS dma
        ON c.fk_dma_id = dma.dma_id
    LEFT OUTER JOIN prod.detection.input_source inps
        ON c.fk_input_source_id = inps.input_source_id
    LEFT OUTER JOIN inscape_station_map_dedupe AS map
        ON map.mapped_vendor_station_id = COALESCE(c.tms_tuner_channel_id, c.tms_station_id)
        AND map.mapped_vendor = 'TMS'
    LEFT OUTER JOIN prod.detection.epg_station AS station
        ON station.station_id = COALESCE(c.tms_tuner_channel_id, c.tms_station_id)
        AND station.vendor_name = 'TMS'
    LEFT OUTER JOIN prod.detection.epg_show AS show
        ON show.show_id = COALESCE(c.tms_tuner_program_id, c.tms_show_id)
        AND show.vendor_name = 'TMS'
    LEFT OUTER JOIN prod.detection.epg_show AS backup_show
        ON backup_show.show_id = COALESCE(c.tuner_program_id, c.fk_show_id)
        AND COALESCE(c.tms_tuner_program_id, c.tms_show_id) IS NULL
       AND backup_show.vendor_name = 'TIVO'
       AND 'comscore' != 'nielsen'
    LEFT OUTER JOIN prod.detection.epg_station AS backup_station
        ON backup_station.station_id = COALESCE(c.tuner_channel_id, c.fk_station_id)
        AND COALESCE(c.tms_tuner_channel_id, c.tms_station_id) IS NULL
        AND backup_station.vendor_name = 'TIVO'
        AND 'comscore' != 'nielsen'
    LEFT OUTER JOIN inscape_station_map_dedupe AS backup_map
        ON backup_map.mapped_vendor_station_id = COALESCE(c.tuner_channel_id, c.fk_station_id)
        AND COALESCE(c.tms_tuner_channel_id, c.tms_station_id) IS NULL
        AND backup_map.mapped_vendor = 'TIVO'
        AND 'comscore' != 'nielsen'
    LEFT OUTER JOIN station_distribution_blacklist AS station_blacklist
        ON COALESCE(c.tms_tuner_channel_id, c.tms_station_id) = station_blacklist.station_id
        AND station_blacklist.vendor_name = map.mapped_vendor
    LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS station_obfs
        ON COALESCE(c.tms_tuner_channel_id, c.tms_station_id) = station_obfs.vendor_station_id
        AND station_obfs.vendor_name = map.mapped_vendor 
    LEFT OUTER JOIN prod.detection.vizio_epg_station AS vizio_station
        ON c.vizio_epg_station = vizio_station.station_id
    LEFT OUTER JOIN prod.detection.vizio_epg_program_aggregate AS vizio_program
        ON CAST(c.vizio_epg_program AS BIGINT) = vizio_program.program_aggregate_id
        AND c.vizio_epg_program != '0'
    LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
        ON cid.content_id = c.fk_content_id
    LEFT OUTER JOIN prod.detection.content_id_external_firehose AS m
        ON m.fk_content_id = c.fk_content_id
    LEFT OUTER JOIN prod.detection.clients cl
        ON m.fk_client_id = cl.client_id
        AND cl.client_name <> 'comscore'
    LEFT OUTER JOIN prod.detection.content_id_external_firehose AS md
        ON md.fk_content_id = c.fk_content_id
    LEFT OUTER JOIN prod.detection.clients cli
        ON md.fk_client_id = cli.client_id
        AND cli.client_name = 'comscore'
    LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
        ON c.session_start >= tvis.create_timestamp
        AND c.session_start < tvis.next_create_timestamp
        -- AND tvis.create_timestamp <= '2025-03-25 19:00:00'::timestamp
        -- AND tvis.next_create_timestamp >= '2025-03-18 17:00:00'::timestamp
        AND tvis.fk_tvid = 160189140
        AND  c.fk_tvid = tvis.fk_tvid
        AND  c.fk_input_source_id = tvis.fk_input_source_id
    LEFT OUTER JOIN prod.detection.tv_inputsource tis
        ON  c.session_start >= (tis.create_timestamp::double)::timestamp  
        AND c.session_start < (tis.next_create_timestamp::double)::timestamp
        AND c.fk_tvid = tis.fk_tvid
        AND tis.fk_tvid = 160189140
        AND c.fk_input_source_id = tis.fk_input_source_id
        AND tis.create_timestamp <= ('2025-03-25 19:00:00'::timestamp::double)::timestamp
        AND tis.next_create_timestamp >= ('2025-03-18 17:00:00'::timestamp::double)::timestamp
    LEFT OUTER JOIN activity_obfuscation AS appb
        ON tis.app_name = appb.app_name
    LEFT OUTER JOIN viewing_obfuscation AS acrb
        ON tis.app_name = acrb.app_name
    LEFT OUTER JOIN
        prod.detection.free_channels_distribution_blacklist chanb
        ON vizio_station.name = chanb.channel_name
    -- LEFT OUTER JOIN
    --     prod.detection.tv_ip_address AS ip
    --     ON c.session_start >= ip.create_timestamp
    --     AND c.session_start < ip.next_create_timestamp
    --     AND ip.create_timestamp <= '2025-03-25 19:00:00'::timestamp
    --     AND ip.next_create_timestamp >= '2025-03-18 17:00:00'::timestamp
    --     AND tv.tvid = ip.fk_tvid
    --     AND ip.fk_tvid = 160189140
    LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS nielsen_blacklist
        ON nielsen_blacklist.station_id = COALESCE(map.inscape_station_id, backup_map.inscape_station_id)
        AND c.session_start >= nielsen_blacklist.blacklist_start
        AND c.session_start < nielsen_blacklist.blacklist_end
        AND 'comscore' != 'nielsen' 
    LEFT OUTER JOIN prod.detection.nielsen_replacement_local AS rep_local
        ON COALESCE(map.inscape_station_id, backup_map.inscape_station_id) = rep_local.station_id 
        AND COALESCE(c.airdate, c.tms_airdate) = rep_local.airdate
        AND COALESCE(c.fk_show_id, c.tms_show_id, c.tms_tuner_program_id, c.tuner_program_id) = rep_local.fk_show_id
        AND c.fk_dma_id = rep_local.dma_id
        AND 'comscore' != 'nielsen'
    LEFT OUTER JOIN prod.detection.nielsen_replacement_national_nyc AS rep_nyc_nat
        ON COALESCE(map.inscape_station_id, backup_map.inscape_station_id) = rep_nyc_nat.station_id
        AND COALESCE(c.airdate, c.tms_airdate) = rep_nyc_nat.airdate
        AND COALESCE(c.fk_show_id, c.tms_show_id, c.tms_tuner_program_id, c.tuner_program_id) = rep_nyc_nat.fk_show_id
        AND 'comscore' != 'nielsen'
    WHERE
        c.fk_tvid = 160189140
  AND (
       (c.session_start >= '2025-03-18 17:00:00' AND c.session_start < '2025-03-18 22:00:00')
    OR (c.session_start >= '2025-03-20 18:00:00' AND c.session_start < '2025-03-20 21:00:00')
    OR (c.session_start >= '2025-03-25 16:00:00' AND c.session_start < '2025-03-25 19:00:00')
    )
    AND CASE c.file_ingested
        WHEN true THEN
            CASE NULLIF(SPLIT(cid.content_cid, '_')[1], '') IS NOT NULL AND NULLIF(SPLIT(cid.content_cid, '_')[2], '') IS NULL
            WHEN true THEN SPLIT(cid.content_cid, '_')[1]
            ELSE NULL
            END
        ELSE COALESCE(station.station_call_sign, backup_station.station_call_sign, 'KeepSessionForNullReport')
        END NOT IN (SELECT DISTINCT chan_callsign FROM customer_reports.bad_chan_callsign)
ORDER BY c.session_start;

In [0]:
%sql
SELECT vc.*
FROM prod.detection.viewing_content_firehose vc
WHERE vc.fk_tvid = 160189140
  AND (
       (vc.session_start >= '2025-03-18 17:00:00' AND vc.session_start < '2025-03-18 22:00:00')
    OR (vc.session_start >= '2025-03-20 18:00:00' AND vc.session_start < '2025-03-20 21:00:00')
    OR (vc.session_start >= '2025-03-25 16:00:00' AND vc.session_start < '2025-03-25 19:00:00')
    )
ORDER BY vc.session_start

In [0]:
%sql
SELECT *
FROM prod.detection.tv_input_stats_firehose tvis
WHERE tvis.create_timestamp <= '2025-03-25 19:00:00'::timestamp
        AND tvis.next_create_timestamp >= '2025-03-18 17:00:00'::timestamp
        AND tvis.fk_tvid = 160189140

In [0]:
%sql
WITH inscape_map_deduped AS (
SELECT 
        inscape_station_id, 
        inscape_call_sign, 
        mapped_vendor,
        mapped_vendor_station_id
    FROM (
        SELECT 
          inscape_station_id, 
          inscape_call_sign, 
          mapped_vendor, 
          mapped_vendor_station_id,
          ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        FROM detection.inscape_station_map
      ) ism
    WHERE ism.rn = 1
)
SELECT vc.fk_tvid,
       vc.fk_show_id,
       tivo_ism.mapped_vendor_station_id AS fk_station_id,
       vc.session_duration,
       vc.airdate,
       vc.session_start,
       vc.session_end,
       vc.media_time_start, 
       vc.media_time_end,
       vc.runtime,
       vc.is_live,
       tivo_tuner_sch_lat.fk_station_id AS tuner_channel_id,
       tivo_tuner_sch_lat.schedule_id   AS tuner_schedule_id,
       tivo_tuner_sch_lat.fk_show_id    AS tuner_program_id,
       vc.fk_station_id                  AS tms_station_id,
       tms_sch_lat.fk_show_id           AS tms_show_id,
       tms_sch_lat.airdate              AS tms_airdate,
       tms_sch.schedule_id              AS tms_schedule_id,
       NULLIF(vc.tuner_channel_id,98989898)        AS tms_tuner_channel_id,
       vc.tuner_schedule_id                        AS tms_tuner_schedule_id,
       NULLIF(vc.tuner_program_id,98989898)        AS tms_tuner_program_id,
       vc.tuner_channel_number                     AS tuner_channel_number,
       vc.partition_key,
       vc.input_file_name
      FROM detection.viewing_content_firehose vc
      LEFT JOIN inscape_map_deduped tivo_ism
        ON tivo_ism.inscape_station_id = vc.fk_station_id
        AND tivo_ism.mapped_vendor = 'TIVO'
      LEFT JOIN detection.epg_schedule_latest AS tms_sch_lat
        ON tms_sch_lat.fk_station_id = vc.fk_station_id
        AND tms_sch_lat.vendor_name = 'TMS'
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) > tms_sch_lat.airdate
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) <= tms_sch_lat.airdate_end
      LEFT JOIN detection.epg_schedule tms_sch
        ON tms_sch.airdate = tms_sch_lat.airdate
        AND tms_sch.fk_show_id = tms_sch_lat.fk_show_id
        AND tms_sch.fk_station_id = tms_sch_lat.fk_station_id
        AND DATE(tms_sch.airdate) >= CURRENT_DATE - INTERVAL 120 DAY
        AND tms_sch.airdate <= CURRENT_DATE + INTERVAL 2 DAY
      LEFT JOIN inscape_map_deduped AS tivo_tuner_map
        ON tivo_tuner_map.inscape_station_id = vc.tuner_channel_id
        AND tivo_tuner_map.mapped_vendor = 'TIVO'
      LEFT JOIN detection.epg_schedule_latest AS tivo_tuner_sch_lat
          ON tivo_tuner_sch_lat.fk_station_id = tivo_tuner_map.mapped_vendor_station_id
        AND tivo_tuner_sch_lat.vendor_name = 'TIVO'
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) > tivo_tuner_sch_lat.airdate
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) <= tivo_tuner_sch_lat.airdate_end
      WHERE vc.session_start < vc.session_end
        AND vc.fk_tvid = 164267145
        AND vc.session_start >= '2025-03-25 21:00:00'
        AND vc.session_start < '2025-03-25 22:00:00'
      ORDER BY session_start,fk_tvid,fk_station_id,fk_show_id,fk_dma_id,fk_schedule_id;

In [0]:
%sql
SELECT * FROM detection.viewing_content_firehose WHERE fk_tvid=164267145 AND session_start BETWEEN '2025-03-25 21:00:00' and '2025-03-25 22:00:00' 

In [0]:
%sql
SELECT sh.title, sch.airdate, st.station_call_sign
FROM detection.epg_schedule sch
JOIN detection.epg_show sh
  ON sh.show_id = sch.fk_show_id
JOIN detection.epg_station st
  ON st.station_id = sch.fk_station_id
WHERE sch.vendor_name = 'TIVO'
  AND timestampadd(SECOND, 800, '2025-03-25 16:00:00') > sch.airdate
  AND timestampadd(SECOND, 800, '2025-03-25 16:00:00') <= TIMESTAMPADD(SECOND, sch.duration, sch.airdate)
  AND st.station_call_sign = 'KOPBDT'

In [0]:
%sql
WITH activity_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_activity_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
, viewing_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, linear_sh.title AS linear_title
, linear_st.inscape_station_name as linear_station_name
, linear_st.station_call_sign AS    linear_call_sign
, tuner_sh.title AS tuner_title
, tuner_st.inscape_station_name as tuner_station_name
, tuner_st.station_call_sign AS    tuner_call_sign
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT') THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND cid.content_cid != 'unknown' THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name ='WatchFree+' THEN 'APPS'
       ELSE tvis.category
  END AS input_category
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL THEN 'OTA'
       WHEN inps.input_source = 'DTV' THEN 'OTA'
       ELSE tvis.input_device
  END AS input_device
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is not null AND (input_device IS NULL OR input_device = 'OTA') THEN 'WatchFree+'
       WHEN UPPER(tvis.category) = 'APPS' THEN
           CASE WHEN vc.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                WHEN vc.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                WHEN appb.app_name IS NOT NULL THEN 'OBFUSCATED'
                WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND cid.content_cid <> 'unknown' THEN NULL
                WHEN lower(tis.app_name) = 'unknown' THEN NULL
                ELSE tis.app_name END
         WHEN vc.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') AND COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is NULL THEN 'vMVPD'
         WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV') AND (input_device IS NULL OR input_device = 'OTA') AND tis.app_name ='WatchFree+' AND vc.is_live = TRUE THEN 'WatchFree+'
    END AS app_name
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station linear_st
  ON linear_st.station_id = vc.fk_station_id
LEFT JOIN detection.epg_show linear_sh
  ON linear_sh.show_id = vc.fk_show_id
LEFT JOIN detection.epg_station tuner_st
  ON tuner_st.station_id = vc.tuner_channel_id
LEFT JOIN detection.epg_show tuner_sh
  ON tuner_sh.show_id = vc.tuner_program_id
LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
  ON vc.session_start >= tvis.create_timestamp
 AND vc.session_start < tvis.next_create_timestamp
 AND vc.fk_tvid = tvis.fk_tvid
 AND vc.fk_input_source_id = tvis.fk_input_source_id
LEFT OUTER JOIN prod.detection.tv_inputsource tis
  ON vc.session_start >= (tis.create_timestamp::double)::timestamp  
 AND vc.session_start < (tis.next_create_timestamp::double)::timestamp
 AND vc.fk_tvid = tis.fk_tvid
 AND vc.fk_input_source_id = tis.fk_input_source_id
LEFT OUTER JOIN prod.detection.input_source inps
  ON vc.fk_input_source_id = inps.input_source_id
LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
  ON cid.content_id = vc.fk_content_id
LEFT OUTER JOIN activity_obfuscation AS appb
  ON tis.app_name = appb.app_name
WHERE vc.fk_tvid=162991395 AND vc.session_start BETWEEN '2025-03-25 17:00:00' and '2025-03-25 18:00:00' 
ORDER BY vc.session_start


In [0]:
%sql
WITH activity_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_activity_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
, viewing_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, linear_sh.title AS linear_title
, linear_st.inscape_station_name as linear_station_name
, linear_st.station_call_sign AS    linear_call_sign
, tuner_sh.title AS tuner_title
, tuner_st.inscape_station_name as tuner_station_name
, tuner_st.station_call_sign AS    tuner_call_sign
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT') THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND cid.content_cid != 'unknown' THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name ='WatchFree+' THEN 'APPS'
       ELSE tvis.category
  END AS input_category
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL THEN 'OTA'
       WHEN inps.input_source = 'DTV' THEN 'OTA'
       ELSE tvis.input_device
  END AS input_device
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is not null AND (input_device IS NULL OR input_device = 'OTA') THEN 'WatchFree+'
       WHEN UPPER(tvis.category) = 'APPS' THEN
           CASE WHEN vc.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                WHEN vc.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                WHEN appb.app_name IS NOT NULL THEN 'OBFUSCATED'
                WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND cid.content_cid <> 'unknown' THEN NULL
                WHEN lower(tis.app_name) = 'unknown' THEN NULL
                ELSE tis.app_name END
         WHEN vc.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') AND COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is NULL THEN 'vMVPD'
         WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV') AND (input_device IS NULL OR input_device = 'OTA') AND tis.app_name ='WatchFree+' AND vc.is_live = TRUE THEN 'WatchFree+'
    END AS app_name
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station linear_st
  ON linear_st.station_id = vc.fk_station_id
LEFT JOIN detection.epg_show linear_sh
  ON linear_sh.show_id = vc.fk_show_id
LEFT JOIN detection.epg_station tuner_st
  ON tuner_st.station_id = vc.tuner_channel_id
LEFT JOIN detection.epg_show tuner_sh
  ON tuner_sh.show_id = vc.tuner_program_id
LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
  ON vc.session_start >= tvis.create_timestamp
 AND vc.session_start < tvis.next_create_timestamp
 AND vc.fk_tvid = tvis.fk_tvid
 AND vc.fk_input_source_id = tvis.fk_input_source_id
LEFT OUTER JOIN prod.detection.tv_inputsource tis
  ON vc.session_start >= (tis.create_timestamp::double)::timestamp  
 AND vc.session_start < (tis.next_create_timestamp::double)::timestamp
 AND vc.fk_tvid = tis.fk_tvid
 AND vc.fk_input_source_id = tis.fk_input_source_id
LEFT OUTER JOIN prod.detection.input_source inps
  ON vc.fk_input_source_id = inps.input_source_id
LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
  ON cid.content_id = vc.fk_content_id
LEFT OUTER JOIN activity_obfuscation AS appb
  ON tis.app_name = appb.app_name
WHERE vc.fk_tvid=164267145 AND vc.session_start BETWEEN '2025-03-25 21:00:00' and '2025-03-25 22:00:00' 
ORDER BY vc.session_start


In [0]:
%sql
WITH activity_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_activity_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
, viewing_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = 'comscore'
    WHERE override.app_name IS NULL
)
SELECT vc.fk_tvid
, vc.session_start
, vc.session_end
, vc.session_duration
, linear_sh.title AS linear_title
, linear_st.inscape_station_name as linear_station_name
, linear_st.station_call_sign AS    linear_call_sign
, tuner_sh.title AS tuner_title
, tuner_st.inscape_station_name as tuner_station_name
, tuner_st.station_call_sign AS    tuner_call_sign
, vc.is_live
, CASE WHEN vc.tuner_channel_id IS NOT NULL THEN 'TUNER' ELSE 'ACR' END AS viewing_type
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT') THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND cid.content_cid != 'unknown' THEN 'HD TV'
       WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name ='WatchFree+' THEN 'APPS'
       ELSE tvis.category
  END AS input_category
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) IS NOT NULL THEN 'OTA'
       WHEN inps.input_source = 'DTV' THEN 'OTA'
       ELSE tvis.input_device
  END AS input_device
, CASE WHEN COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is not null AND (input_device IS NULL OR input_device = 'OTA') THEN 'WatchFree+'
       WHEN UPPER(tvis.category) = 'APPS' THEN
           CASE WHEN vc.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                WHEN vc.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                WHEN appb.app_name IS NOT NULL THEN 'OBFUSCATED'
                WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND cid.content_cid <> 'unknown' THEN NULL
                WHEN lower(tis.app_name) = 'unknown' THEN NULL
                ELSE tis.app_name END
         WHEN vc.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') AND COALESCE(vc.tuner_channel_id, vc.tms_tuner_channel_id) is NULL THEN 'vMVPD'
         WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV') AND (input_device IS NULL OR input_device = 'OTA') AND tis.app_name ='WatchFree+' AND vc.is_live = TRUE THEN 'WatchFree+'
    END AS app_name
FROM detection.viewing_content_firehose vc
LEFT JOIN detection.epg_station linear_st
  ON linear_st.station_id = vc.tms_station_id
LEFT JOIN detection.epg_show linear_sh
  ON linear_sh.show_id = vc.tms_show_id
LEFT JOIN detection.epg_station tuner_st
  ON tuner_st.station_id = vc.tms_tuner_channel_id
LEFT JOIN detection.epg_show tuner_sh
  ON tuner_sh.show_id = vc.tms_tuner_program_id
LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
  ON vc.session_start >= tvis.create_timestamp
 AND vc.session_start < tvis.next_create_timestamp
 AND vc.fk_tvid = tvis.fk_tvid
 AND vc.fk_input_source_id = tvis.fk_input_source_id
LEFT OUTER JOIN prod.detection.tv_inputsource tis
  ON vc.session_start >= (tis.create_timestamp::double)::timestamp  
 AND vc.session_start < (tis.next_create_timestamp::double)::timestamp
 AND vc.fk_tvid = tis.fk_tvid
 AND vc.fk_input_source_id = tis.fk_input_source_id
LEFT OUTER JOIN prod.detection.input_source inps
  ON vc.fk_input_source_id = inps.input_source_id
LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
  ON cid.content_id = vc.fk_content_id
LEFT OUTER JOIN activity_obfuscation AS appb
  ON tis.app_name = appb.app_name
WHERE vc.fk_tvid=175833069 AND vc.session_start BETWEEN '2025-03-25 14:00:00' and '2025-03-25 22:00:00' 
ORDER BY vc.session_start


In [0]:
%sql
SELECT NVL(c.tuner_channel_id, c.tms_tuner_channel_id) IS NULL AS null_tuner_station_id
, c.fk_station_id IS NULL AS null_linear_Station_id_null
, c.vizio_epg_station IS NULL AS null_vizio_epg_station
,     CASE
        WHEN c.vizio_epg_station IS NOT NULL THEN 't'
        WHEN COALESCE(NULLIF(c.tms_tuner_channel_id,98989898), c.tuner_channel_id) IS NOT NULL THEN 't'
        WHEN 'comscore' != 'nielsen' AND COALESCE(c.tms_airdate,c.airdate) IS NULL THEN NULL
        WHEN 'comscore' = 'nielsen' AND c.tms_airdate IS NULL THEN NULL
        WHEN inps.input_source = 'DTV' OR tvis.input_device = 'OTA' THEN 't'
    ELSE CASE WHEN c.is_live = TRUE THEN 't' WHEN c.is_live = FALSE THEN 'f' ELSE NULL END
    END cure_is_live
, is_live
, COUNT(*)
FROM detection.viewing_content_firehose c
LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
  ON c.session_start >= tvis.create_timestamp
 AND c.session_start < tvis.next_create_timestamp
 AND c.fk_tvid = tvis.fk_tvid
 AND c.fk_input_source_id = tvis.fk_input_source_id
 AND tvis.next_create_timestamp >= CURRENT_DATE - 1
LEFT OUTER JOIN prod.detection.tv_inputsource tis
  ON c.session_start >= (tis.create_timestamp::double)::timestamp  
 AND c.session_start < (tis.next_create_timestamp::double)::timestamp
 AND c.fk_tvid = tis.fk_tvid
 AND c.fk_input_source_id = tis.fk_input_source_id
 AND tis.next_create_timestamp >= CURRENT_DATE - 1
LEFT OUTER JOIN prod.detection.input_source inps
  ON c.fk_input_source_id = inps.input_source_id
LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
  ON cid.content_id = c.fk_content_id
WHERE session_start >= CURRENT_DATE - 1
AND session_start < CURRENT_DATE
GROUP BY 1, 2, 3, 4, 5

In [0]:
%sql
SELECT CASE WHEN NVL(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN 'Tuner Session Present' ELSE 'Tuner Null' END AS null_tuner_station_id
, CASE WHEN inps.input_source = 'DTV' THEN 'Device name Changed from '||tvis.input_device||' to OTA' END AS input_device_changed_based_on_dtv
, CASE WHEN is_live = FALSE AND inps.input_source = 'OTA' AND tis.app_name ='WatchFree+' THEN 'scenario_1' END AS scenario_1
, tvis.input_device AS unchanged_input_device
, COUNT(*)
FROM detection.viewing_content_firehose c
LEFT OUTER JOIN prod.detection.tv_input_stats_firehose tvis
  ON c.session_start >= tvis.create_timestamp
 AND c.session_start < tvis.next_create_timestamp
 AND c.fk_tvid = tvis.fk_tvid
 AND c.fk_input_source_id = tvis.fk_input_source_id
 AND tvis.next_create_timestamp >= CURRENT_DATE - 1
LEFT OUTER JOIN prod.detection.tv_inputsource tis
  ON c.session_start >= (tis.create_timestamp::double)::timestamp
 AND c.session_start < (tis.next_create_timestamp::double)::timestamp
 AND c.fk_tvid = tis.fk_tvid
 AND c.fk_input_source_id = tis.fk_input_source_id
 AND tis.next_create_timestamp >= CURRENT_DATE - 1
LEFT OUTER JOIN prod.detection.input_source inps
  ON c.fk_input_source_id = inps.input_source_id
-- LEFT OUTER JOIN prod.detection.content_ids_firehose AS cid
--   ON cid.content_id = c.fk_content_id
WHERE session_start >= CURRENT_DATE - 1
  AND session_start < CURRENT_DATE
GROUP BY 1, 2, 3, 4

In [0]:
%sql
SELECT * FROM dev.ashwin.r1411_content_with_null_comscore_2025_03_31_21_production
LIMIT 10

In [0]:
%sql
SELECT * FROM detection.tv_input_stats_firehose
WHERE fk_tvid = 175833069
AND next_create_timestamp >= '2025-03-17T00:00:00'

In [0]:
%sql
SELECT fk_tvid, session_start, session_end, session_duration
, tms_tuner_channel_id, tuner_channel_id
FROM detection.viewing_content_firehose
WHERE fk_tvid = 175833069
AND session_start >= '2025-03-11T00:00:00'
-- AND COALESCE(tms_tuner_channel_id, tuner_channel_id) IS NOT NULL
ORDER BY session_start

In [0]:
%sql
SELECT token FROM detection.tv WHERE tvid = 175833069

In [0]:
%sql
SELECT * FROM detection.tuner_sessionized
WHERE tvid = 175833069
AND session_start >= '2025-03-11T00:00:00'

In [0]:
%sql
SELECT tsld.tvid, tsld.client_version, tsld.client_version_string, tsld.firmware_version, tsld.create_timestamp, tv.model_name FROM prod.detection.tv_firmware_latest_daily tsld
JOIN detection.tv
  ON tv.tvid = tsld.tvid
WHERE tsld.tvid IN (164267145
,164224255
,175833069
,164224401
,162991395)